**Research Objective**: Fine-tuning Gemma-3-270M to act as a specialized decomposition agent for multi-hop queries, distilling logical hop-transition capabilities from GPT-4o.

**This notebook now also hosts the v2 phase 1 benchmark**: score the untrained Gemma-3-270M-it, the fine-tuned sys-arm model, Llama-3.1-8B-Instruct through Groq, and the GPT-4o teacher, all on the same 95 test questions from `data/test_sys.jsonl`, on accuracy (through notebook 03) and on response latency.

In [ ]:
!pip install -q -U transformers trl datasets accelerate huggingface_hub

In [1]:
import torch
import transformers

In [ ]:
!pip install python-dotenv

In [ ]:
from openai import OpenAI
# from google.colab import userdata

# hf_token = userdata.get("HF_TOKEN")
# groq_token = userdata.get("GROQ_API_KEY")

# client = OpenAI(
#     api_key=userdata.get("OPENAI_API_KEY")
# )

# groq_client = OpenAI(
#     api_key=groq_token,
#     base_url="https://api.groq.com/openai/v1"
# )

from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
TEACHER_PROMPT = """You are an analytical reasoning agent specialized in breaking down multi-hop questions.

Your objective is to determine ONLY the first logical step (Hop 1) required to solve the question.

CRITICAL CONSTRAINTS:
1. DO NOT answer the question. Stop reasoning immediately after formulating the first hop.
2. Output your response strictly in YAML format. Do not include introductory or concluding text, markdown formatting, or conversational filler.
3. TARGET EXACT ANCHORS: The `target_entity` MUST be taken verbatim from the question. Prefer the explicitly named entity that anchors the lookup. Only when the question names no entity — when it refers to its subject by description alone — use that description verbatim. Never canonicalize, resolve, guess, or inject outside knowledge: do not expand a partial name to its full form, and do not replace a description with the real-world entity it denotes.
4. SEPARATE THE UNKNOWN: Anything you are trying to find out — whether a missing entity or a property — belongs entirely in the `thought`. The `target_entity` is only the known starting bridge the question hands you.
5. STRICT QUOTING: Every value in your YAML output (`thought`, `action`, and `target_entity`) MUST be wrapped in double quotes.

YAML SCHEMA:
thought: "[Your logical deduction of what needs to be found first about the anchor]"
action: "Lookup"
target_entity: "[The exact anchor string handed to you by the question]"

EXAMPLE 1 (Named Entity):
Question: The Oberoi family is part of a hotel company that has a head office in what city?
thought: "I need to find which hotel company the Oberoi family belongs to."
action: "Lookup"
target_entity: "Oberoi family"

EXAMPLE 2 (Comparison):
Question: Were Scott Derrickson and Ed Wood of the same nationality?
thought: "I need to find the nationality of Scott Derrickson first to eventually compare it to Ed Wood."
action: "Lookup"
target_entity: "Scott Derrickson"

EXAMPLE 3 (Anchor Move / Relational):
Question: The wife of Arthur Miller starred in what movie?
thought: "I need to find out who the wife of Arthur Miller is first."
action: "Lookup"
target_entity: "Arthur Miller"

EXAMPLE 4 (Property):
Question: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
thought: "I need to find the chemical that Cadmium Chloride is slightly soluble in."
action: "Lookup"
target_entity: "Cadmium Chloride"

EXAMPLE 5 (Pure Descriptive Anchor):
Question: What language is most widely spoken in the most populous country in Africa?
thought: "I need to identify which country is the most populous in Africa first."
action: "Lookup"
target_entity: "the most populous country in Africa"
"""

print(TEACHER_PROMPT)

You are an analytical reasoning agent specialized in breaking down multi-hop questions.

Your objective is to determine ONLY the first logical step (Hop 1) required to solve the question.

CRITICAL CONSTRAINTS:
1. DO NOT answer the question. Stop reasoning immediately after formulating the first hop.
2. Output your response strictly in YAML format. Do not include introductory or concluding text, markdown formatting, or conversational filler.
3. TARGET EXACT ANCHORS: The `target_entity` MUST be taken verbatim from the question. Prefer the explicitly named entity that anchors the lookup. Only when the question names no entity — when it refers to its subject by description alone — use that description verbatim. Never canonicalize, resolve, guess, or inject outside knowledge: do not expand a partial name to its full form, and do not replace a description with the real-world entity it denotes.
4. SEPARATE THE UNKNOWN: Anything you are trying to find out — whether a missing entity or a prope

# 1. Untrained baseline — load Gemma-3-270M-it

In [3]:
MODEL_NAME = "google/gemma-3-270m-it"

In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] Using device: {device}")

[INFO] Using device: cuda


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

untrained_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    device_map="auto",
    attn_implementation="eager",
    # torch_dtype = torch.float32
)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"[INFO] Model on devices: {untrained_model.device}")
print(f"[INFO] Model using dtypes: {untrained_model.dtype}")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

[INFO] Model on devices: cuda:0
[INFO] Model using dtypes: torch.float32


### attn_implementation = 'sdpa'

In [ ]:
untrained_sdpa = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    device_map="auto",
    attn_implementation="sdpa",
)

## 1.5 Retrain the sys arm in this runtime

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train_sys.jsonl to train_sys.jsonl


In [ ]:
train_ds = load_dataset(
    "json", data_files="train_sys.jsonl", split="train"
).select_columns(['messages'])

print(train_ds)
print(train_ds[0]["messages"])

Dataset({
    features: ['messages'],
    num_rows: 379
})
[{'role': 'system', 'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'}, {'role': 'user', 'content': 'Pearl Lowe and Alison Goldfrapp, is of which nationality?'}, {'role': 'assistant', 'content': 'thought: "I need to find the nationality of Pearl Lowe first."\naction: "Lookup"\ntarget_entity: "Pearl Lowe"'}]


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

trained_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    device_map="auto",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="gemma3-270m-hop1-sys",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    max_length=256,
    fp16=True,
    assistant_only_loss=True,
    logging_steps=10,
    report_to="none",
)

from trl import SFTTrainer

trainer = SFTTrainer(
    model=trained_model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.560403
20,0.440201
30,0.349052
40,0.366379
50,0.252770
60,0.189469
70,0.138228
80,0.148888
90,0.153470
100,0.114351


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=144, training_loss=0.2751341540780332, metrics={'train_runtime': 128.0379, 'train_samples_per_second': 8.88, 'train_steps_per_second': 1.125, 'total_flos': 94106806268160.0, 'train_loss': 0.2751341540780332, 'entropy': 0.06464770808815956, 'num_tokens': 123897.0, 'mean_token_accuracy': 0.9835876971483231, 'epoch': 3.0})

In [ ]:
trained_model.save_pretrained("/content/gemma3-270m-hop1-sys")
tokenizer.save_pretrained("/content/gemma3-270m-hop1-sys")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/gemma3-270m-hop1-sys/tokenizer_config.json',
 '/content/gemma3-270m-hop1-sys/chat_template.jinja',
 '/content/gemma3-270m-hop1-sys/tokenizer.json')

In [ ]:
from google.colab import files
files.download("/content/gemma3-270m-hop1-sys/model.safetensors")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 2. Phase 1 benchmark — comparison across four models

## 2.1 Load the 95 test questions

In [7]:
from google.colab import files
uploaded = files.upload()

Saving test_sys.jsonl to test_sys.jsonl


I looked at the model's own configuration here, to understand why generation kept coming back empty under the full `TEACHER_PROMPT` later in this section.

In [19]:
untrained_model.config

Gemma3TextConfig {
  "_sliding_window_pattern": 6,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "dtype": "float32",
  "eos_token_id": 1,
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 640,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "layer_types": [
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "model_type": "gemma3_text",
  "num_attention_heads": 4,
  "num_hi

In [17]:
from datasets import load_dataset

test_rows = load_dataset("json", data_files="test_sys.jsonl", split="train")

exm_hop_1 = test_rows[0]['messages']
print(f"[INFO] Example hop 1: {exm_hop_1}")

[INFO] Example hop 1: [{'role': 'system', 'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'}, {'role': 'user', 'content': 'Robinsons and Pocari Sweat are both what kind of product?'}, {'role': 'assistant', 'content': 'thought: "I need to find out what kind of product Robinsons is."\naction: "Lookup"\ntarget_entity: "Robinsons"'}]


In [ ]:
def call_untrained_teacherprompt(message) -> str:
  input_message = [
      {"role": "system", "content": TEACHER_PROMPT},
      message[1]
  ]
  inputs = tokenizer.apply_chat_template(
      input_message,
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt"
  )


  inputs = inputs.to(untrained_model.device)
  outputs = untrained_model.generate(
      **inputs,
      max_new_tokens = 150,
      do_sample=False
  )
  inputs_length = inputs['input_ids'].shape[1]
  return tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)

In [ ]:

def call_gemma_270M(message) -> str:

  inputs = tokenizer.apply_chat_template(
      message[:-1],
      add_generation_prompt=True,
      return_tensors="pt",
      tokenize=True
  )
  inputs = inputs.to(untrained_model.device)
  outputs = untrained_model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=False
  )
  inputs_length = inputs['input_ids'].shape[1]
  return tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)


def call_gemma_270M_sdpa(message) -> str:

  inputs = tokenizer.apply_chat_template(
      message[:-1],
      add_generation_prompt=True,
      return_tensors="pt",
      tokenize=True
  )
  inputs = inputs.to(untrained_sdpa.device)
  outputs = untrained_sdpa.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=False
  )
  inputs_length = inputs['input_ids'].shape[1]
  return tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)


def call_trained_gemma_270M(message) -> str:

  inputs = tokenizer.apply_chat_template(
      message[:-1],
      add_generation_prompt=True,
      return_tensors="pt",
      tokenize=True
  )
  inputs = inputs.to(trained_model.device)
  outputs = trained_model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=False
  )
  inputs_length = inputs['input_ids'].shape[1]
  return tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)


def call_groq(message):
  api_messages = [
     {"role": "system", "content": TEACHER_PROMPT},
     message[1]
  ]
  resp = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=api_messages,
      temperature=0.0
  )
  return resp.choices[0].message.content


def call_gpt4o(message) -> str:
    api_messages = [
     {"role": "system", "content": TEACHER_PROMPT},
     message[1]
  ]
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=api_messages,
        temperature=0.0
    )
    return response.choices[0].message.content

## 2.2 `generate_and_time` + one `call_fn` per backend

In [20]:
import time
import json
from tqdm.auto import tqdm

def generate_and_time(call_fn, message):
  start = time.perf_counter()
  pred_yaml = call_fn(message)
  latency = time.perf_counter() - start
  return pred_yaml, latency


def run_benchmark(call_fn, test_rows):
  results = []
  for row in tqdm(test_rows, desc="Running benchmark"):
    pred_yaml, latency = generate_and_time(call_fn, row['messages'])
    results.append({**row, "pred_yaml": pred_yaml, "latency_s": latency})
  return results

## 2.3 Run all four, save predictions + latency

In [ ]:
pred_eager, lat_eager = generate_and_time(call_gemma_270M, exm_hop_1)
pred_sdpa, lat_sdpa = generate_and_time(call_gemma_270M_sdpa, exm_hop_1)

print("eager:", lat_eager, repr(pred_eager))
print("sdpa: ", lat_sdpa, repr(pred_sdpa))
print("same output:", pred_eager == pred_sdpa)

In [22]:
results = run_benchmark(call_untrained_teacherprompt, test_rows)


with open("preds_untrained_teacherprompt_bench.jsonl", "w") as f:
  for row in results:
    f.write(json.dumps(row) + "\n")

Running benchmark:   0%|          | 0/95 [00:00<?, ?it/s]

In [24]:
files.download("preds_untrained_teacherprompt_bench.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
with open("preds_untrained_teacherprompt_bench.jsonl", "r") as f:
  sample = [json.loads(next(f)) for _ in range(2)]

sample

[{'id': '5ab488f85542990594ba9c58',
  'question': 'Robinsons and Pocari Sweat are both what kind of product?',
  'type': 'comparison',
  'level': 'medium',
  'gold_titles': ['Pocari Sweat', 'Robinsons (drink)'],
  'thought': 'I need to find out what kind of product Robinsons is.',
  'action': 'Lookup',
  'target_entity': 'Robinsons',
  'judge_reason': "Looking up 'Robinsons' is correct as it is one of the products in question and matches the gold title 'Robinsons (drink)'.",
  'messages': [{'role': 'system',
    'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'},
   {'role': 'user',
    'content': 'Robinsons and Pocari Sweat are both what kind of product?'},
   {'role': 'assistant',
    'content': 'thought: "I need to find out what kind of product Robinsons is."\naction: "Lookup"\ntarget_entity: "Robinsons"'}],
  'pred_yaml': '',
  'latency_s': 0.11649575299998105},
 {'id':

In [ ]:
results = run_benchmark(call_groq, test_rows)


with open("preds_groq_bench.jsonl", "w") as f:
  for row in results:
    f.write(json.dumps(row) + "\n")

Running benchmark:   0%|          | 0/95 [00:00<?, ?it/s]

In [ ]:
results = run_benchmark(call_gpt4o, test_rows)


with open("preds_gpt4o_bench.jsonl", "w") as f:
  for row in results:
    f.write(json.dumps(row) + "\n")

In [ ]:
results = run_benchmark(call_trained_gemma_270M, test_rows)


with open("preds_trained_bench.jsonl", "w") as f:
  for row in results:
    f.write(json.dumps(row) + "\n")

Running benchmark:   0%|          | 0/95 [00:00<?, ?it/s]

In [ ]:
with open("preds_trained_bench.jsonl", "r") as f:
  sample = [json.loads(next(f)) for _ in range(2)]

sample

[{'messages': [{'role': 'system',
    'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'},
   {'role': 'user',
    'content': 'Robinsons and Pocari Sweat are both what kind of product?'},
   {'role': 'assistant',
    'content': 'thought: "I need to find out what kind of product Robinsons is."\naction: "Lookup"\ntarget_entity: "Robinsons"'}],
  'pred_yaml': 'thought: "I need to find out what kind of product Robinsons is categorized as."\naction: "Lookup"\ntarget_entity: "Robinsons"',
  'latency_s': 5.711616556999616},
 {'messages': [{'role': 'system',
    'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'},
   {'role': 'user',
    'content': 'Padgate is a suburb of an english town on the banks of the River Mersey whose population was estimated to be 208800 in 2016 